# Forecasting pipeline (combined notebook)

This notebook copies Ashita's data preprocessing and baseline-model scripts from `src/` (`ashita_branch`, commits 105c452 and 7ff8b8a, 10 Nov 2026).

The code is hers, with only these changes so it runs as one notebook from this repository:
- `from data_cleaning import df` and `from forecast_episode_duration import ...` are removed, because those objects are already defined in earlier cells.
- `if __name__ == "__main__": main()` is replaced by a plain `main()` call.
- In Section 5, `FEATURES` is renamed `PEAK_FEATURES` so it doesn't overwrite Section 4's `FEATURES`.
- **El Niño / La Niña rule:** episodes are rebuilt from the monthly ICEN with **±0.4 °C for at least 3 consecutive months** (her file used > +0.5 °C and < −0.7 °C). The relabelled file `icen_monthly_pm04.csv` is read through `ICEN_CSV`, and Section 4's ARIMA end test uses the same ±0.4 °C.
- **Section 4** adds two more duration methods, **ETS** and **BSTS**.

Two bigger changes are explained in their sections:
- **Section 1** reads the global arrays in chunks, so it fits in 8 GB of memory.
- **Section 2** runs three times to create the SST files that Sections 3 and 5 read, which aren't in the repo.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Coastal El Niño / La Niña rule used everywhere in this notebook:
# El Niño when ICEN >= +0.4 °C, La Niña when ICEN <= -0.4 °C, for at least 3 consecutive months.
# (The episodes in Ashita's icen_monthly.csv used > +0.5 °C and < -0.7 °C.)
ENSO_WARM, ENSO_COLD, ENSO_MIN_MONTHS = 0.4, -0.4, 3

ICEN_SOURCE = Path("../data/raw/icen_monthly_original.csv")
assert ICEN_SOURCE.exists(), ICEN_SOURCE
icen_relabelled = pd.read_csv(ICEN_SOURCE)


def qualifying_runs(active):
    """Months inside a run of at least ENSO_MIN_MONTHS consecutive True values."""
    run_id = (active != active.shift()).cumsum()
    return active & active.groupby(run_id).transform("size").ge(ENSO_MIN_MONTHS)


warm = qualifying_runs(icen_relabelled["ICEN"] >= ENSO_WARM)
cold = qualifying_runs(icen_relabelled["ICEN"] <= ENSO_COLD)
phase = pd.Series(np.select([warm, cold], ["El Niño", "La Niña"], ""), index=icen_relabelled.index)
in_episode = phase.ne("")
episode_start = in_episode & phase.ne(phase.shift())
icen_relabelled["ENSO"] = phase.where(in_episode, "Neutral")
icen_relabelled["Episode"] = phase.where(in_episode, "No")
icen_relabelled["Episode_ID"] = episode_start.cumsum().where(in_episode)

# The rest of the notebook (Ashita's code) reads the relabelled file
ICEN_CSV = Path("../data/processed/icen_monthly_pm04.csv")
icen_relabelled.to_csv(ICEN_CSV, index=False)

episodes_pm04 = (icen_relabelled.dropna(subset=["Episode_ID"]).groupby("Episode_ID")
                 .agg(phase=("Episode", "first"), start=("Date", "first"), end=("Date", "last"),
                      months=("ICEN", "size"), peak_icen=("ICEN", lambda s: s.abs().max())))
print(f"Episodes with the ±{ENSO_WARM} °C, {ENSO_MIN_MONTHS}-month rule: {len(episodes_pm04)} "
      f"({episodes_pm04['phase'].value_counts().to_dict()})")
print(episodes_pm04.to_string())

Episodes with the ±0.4 °C, 3-month rule: 28 ({'La Niña': 15, 'El Niño': 13})
              phase       start         end  months  peak_icen
Episode_ID                                                    
1.0         La Niña  2001-06-01  2002-01-01       8       1.09
2.0         El Niño  2002-03-01  2002-05-01       3       0.81
3.0         El Niño  2002-10-01  2003-01-01       4       0.79
4.0         La Niña  2003-04-01  2003-08-01       5       1.30
5.0         La Niña  2004-04-01  2004-08-01       5       1.11
6.0         La Niña  2005-02-01  2005-12-01      11       1.23
7.0         La Niña  2006-03-01  2006-05-01       3       0.78
8.0         El Niño  2006-08-01  2007-01-01       6       0.94
9.0         La Niña  2007-03-01  2008-01-01      11       1.64
10.0        El Niño  2008-04-01  2008-09-01       6       1.13
11.0        El Niño  2009-05-01  2009-09-01       5       0.75
12.0        La Niña  2010-07-01  2011-03-01       9       1.25
13.0        La Niña  2011-09-01  2011-12-

## 1. Data cleaning (based on `data_cleaning.py`)

This uses the same filter as her script: 94°W–65°W, 54°S–4°S, 2001–2022, and SST between −2 and 35 °C. The difference is that it reads the global arrays from disk in chunks and saves only the matching readings to `data/cache/forecast_obs.npz`. Her version loads all of them at once (more than 10 GB). The extract is built once, and later runs load the small file.

Not copied: her `valid_velocity.npy` and `valid_sst_velocity.npy` masks. Nothing in this notebook uses them.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

CACHE = Path("../data/cache")
EXTRACT = CACHE / "forecast_obs.npz"
pd.set_option("display.width", 200)

# Study time - 2001 to 2022
START, END = np.datetime64("2001-01-01", "ns"), np.datetime64("2023-01-01", "ns")

# Selected region - Humboldt Region
R = dict(lon=(-94, -65), lat=(-54, -4))


def download_gdp1h(variables):
    """Download any missing Clouddrift gdp1h arrays into data/cache, one variable at a time."""
    missing = [v for v in variables if not (CACHE / f"{v}.npy").exists()]
    if not missing:
        return
    from clouddrift.datasets import gdp1h
    CACHE.mkdir(parents=True, exist_ok=True)
    ds = gdp1h()
    for v in missing:
        print(f"Downloading gdp1h {v} ...")
        np.save(CACHE / f"{v}.npy", ds[v].values)


def build_extract(chunk=2_000_000):
    """Keep valid SST readings in the study region and period, reading the global arrays in chunks."""
    raw = {v: np.load(CACHE / f"{v}.npy", mmap_mode="r") for v in ["lon", "lat", "time", "sst"]}
    ids = np.load(CACHE / "id.npy")
    last_row = np.cumsum(np.load(CACHE / "rowsize.npy"))
    parts = {k: [] for k in ["id", "time", "lon", "lat", "sst_degC"]}

    for a in range(0, len(raw["lon"]), chunk):
        b = a + chunk
        lon, lat = np.asarray(raw["lon"][a:b]), np.asarray(raw["lat"][a:b])
        in_R = ((lon >= R["lon"][0]) & (lon <= R["lon"][1])
                & (lat >= R["lat"][0]) & (lat <= R["lat"][1]))
        rows = np.flatnonzero(in_R)
        if len(rows) == 0:
            continue

        t = np.asarray(raw["time"][a:b])[rows]
        # Convert SST from Kelvin to Celsius
        sst_c = np.asarray(raw["sst"][a:b])[rows] - 273.15
        valid_sst = (t >= START) & (t < END) & np.isfinite(sst_c) & (sst_c >= -2) & (sst_c <= 35)
        rows = rows[valid_sst]

        parts["id"].append(ids[np.searchsorted(last_row, a + rows, side="right")])
        parts["time"].append(t[valid_sst])
        parts["lon"].append(lon[rows])
        parts["lat"].append(lat[rows])
        parts["sst_degC"].append(sst_c[valid_sst])

    np.savez(EXTRACT, **{k: np.concatenate(v) for k, v in parts.items()})


if not EXTRACT.exists():
    download_gdp1h(["lon", "lat", "time", "sst", "id", "rowsize"])
    build_extract()

obs = np.load(EXTRACT)
df = pd.DataFrame({k: obs[k] for k in ["id", "time", "lon", "lat", "sst_degC"]})

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["doy"] = df["time"].dt.dayofyear

print(f"Observations with SST: {len(df):,}")

Observations with SST: 4,081,401


## 2. Preprocessing: SST anomalies against a baseline fitted only on earlier years (`data_preprocessing_2010.py`)

Her script is wrapped in a function, with `TRAIN_END_YEAR` and `OUTPUT_FILE` as its arguments. It runs three times to create the SST files that Sections 3 and 5 read. Each forecast period gets a baseline fitted only on years before it:

| Baseline years | Output | Used for |
|---|---|---|
| 2001–2010 | `sst_forecast_data_2011_2013.csv` | Section 5, forecasts in 2011–2013 |
| 2001–2013 | `sst_forecast_data_2014_2017.csv` | Section 5, forecasts in 2014–2017 |
| 2001–2017 | `sst_forecast_data.csv` | Sections 3 and 5, 2018–2022 |

The 2013 and 2017 runs are a reconstruction. They follow the periods set in `forecast_peak_sst_arima_gam.py`, because the version she actually ran isn't in the repo.

In [3]:
import numpy as np
import pandas as pd
from scipy.interpolate import BSpline


LAT_SPLINE = (-54, -4, 16)
LON_SPLINE = (-94, -65, 10)
CHUNK_SIZE = 20_000

def spline_basis(values, lower, upper, n_basis):
    degree = 3
    inner = np.linspace(lower, upper, n_basis - degree + 1)
    knots = np.r_[[lower] * degree, inner, [upper] * degree]
    return BSpline.design_matrix(
        np.clip(values, lower, upper), knots, degree
    ).toarray()


def design_matrix(lat, lon, doy):
    lat_basis = spline_basis(lat, *LAT_SPLINE)
    lon_basis = spline_basis(lon, *LON_SPLINE)
    spatial = (lat_basis[:, :, None] * lon_basis[:, None, :]).reshape(
        len(lat), -1
    )

    angle = 2 * np.pi * doy / 365.25
    seasonal = np.column_stack([
        np.ones_like(angle),
        np.cos(angle),
        np.sin(angle),
        np.cos(2 * angle),
        np.sin(2 * angle),
    ])

    return (seasonal[:, :, None] * spatial[:, None, :]).reshape(len(lat), -1)

In [4]:
def build_forecast_data(TRAIN_END_YEAR, OUTPUT_FILE):
    # Apply the same SST and geographic filters as the other forecast files.
    in_sst_range = df["sst_degC"].between(5.0, 35.0)
    outside_study_area = (df["lat"] < -40) & (df["lon"] > -71)
    readings = df.loc[in_sst_range & ~outside_study_area].copy()

    # Fit normal SST using only 2001–2010 observations.
    training = readings.loc[readings["year"].between(2001, TRAIN_END_YEAR)]

    units = (
        training.assign(
            cell_lat=np.floor(training["lat"]),
            cell_lon=np.floor(training["lon"]),
        )
        .groupby(["year", "month", "cell_lat", "cell_lon"], as_index=False)
        .agg(
            lat=("lat", "mean"),
            lon=("lon", "mean"),
            doy=("doy", "mean"),
            sst=("sst_degC", "mean"),
        )
    )

    # Give each training year the same total weight.
    units["weight"] = 1 / units.groupby("year")["sst"].transform("size")
    units["weight"] /= units["weight"].mean()

    n_lat, n_lon = LAT_SPLINE[2], LON_SPLINE[2]
    lat_diff = np.diff(np.eye(n_lat), n=2, axis=0)
    lon_diff = np.diff(np.eye(n_lon), n=2, axis=0)
    spatial_penalty = (
        np.kron(lat_diff.T @ lat_diff, np.eye(n_lon))
        + np.kron(np.eye(n_lat), lon_diff.T @ lon_diff)
    )

    n_features = 5 * n_lat * n_lon
    penalty = np.kron(np.eye(5), spatial_penalty) + 1e-6 * np.eye(n_features)

    # Accumulate the fit one year at a time to limit memory use.
    xtwx = np.zeros((n_features, n_features))
    xtwy = np.zeros(n_features)

    for _, year_data in units.groupby("year"):
        x = design_matrix(
            year_data["lat"].to_numpy(),
            year_data["lon"].to_numpy(),
            year_data["doy"].to_numpy(),
        )
        weighted_xt = x.T * year_data["weight"].to_numpy()
        xtwx += weighted_xt @ x
        xtwy += weighted_xt @ year_data["sst"].to_numpy()

    beta = np.linalg.solve(xtwx + penalty, xtwy)

    # Use the fitted 2001–2010 normal to calculate anomalies for all readings.
    lat = readings["lat"].to_numpy()
    lon = readings["lon"].to_numpy()
    doy = readings["doy"].to_numpy()
    baseline = np.empty(len(readings))

    for start in range(0, len(readings), CHUNK_SIZE):
        batch = slice(start, start + CHUNK_SIZE)
        baseline[batch] = design_matrix(lat[batch], lon[batch], doy[batch]) @ beta

    readings["sst_anom"] = readings["sst_degC"] - baseline

    # Average within 2° cells, then combine the usable cells by area.
    readings["cell_lat"] = np.floor(readings["lat"] / 2) * 2
    readings["cell_lon"] = np.floor(readings["lon"] / 2) * 2
    month_keys = ["year", "month"]

    cells = (
        readings.groupby(month_keys + ["cell_lat", "cell_lon"], as_index=False)
        .agg(anomaly=("sst_anom", "mean"), n_readings=("sst_anom", "size"))
    )
    cells = cells.loc[cells["n_readings"].ge(24)].copy()
    cells["area"] = np.cos(np.radians(cells["cell_lat"] + 1))
    cells["weighted_anomaly"] = cells["anomaly"] * cells["area"]

    monthly = cells.groupby(month_keys).agg(
        weighted_sum=("weighted_anomaly", "sum"),
        area_sum=("area", "sum"),
        n_cells=("anomaly", "size"),
    )
    monthly["sst_anom"] = monthly["weighted_sum"] / monthly["area_sum"]
    monthly = monthly[["sst_anom", "n_cells"]]

    # Keep all calendar months so gaps are visible to the forecast script.
    calendar = pd.MultiIndex.from_product(
        [range(2001, 2023), range(1, 13)], names=month_keys
    )
    monthly = monthly.reindex(calendar)
    monthly["n_drifters"] = readings.groupby(month_keys)["id"].nunique()

    count_columns = ["n_cells", "n_drifters"]
    monthly[count_columns] = monthly[count_columns].fillna(0).astype(int)
    monthly["low_coverage"] = (
        monthly["n_cells"].lt(10) | monthly["n_drifters"].lt(5)
    )

    monthly.to_csv(OUTPUT_FILE)
    print(f"Saved {len(monthly)} calendar months to {OUTPUT_FILE}")
    print(f"Months with an SST anomaly: {monthly['sst_anom'].notna().sum()}")

In [5]:
for train_end_year, output_file in [
    (2010, "../data/processed/sst_forecast_data_2011_2013.csv"),
    (2013, "../data/processed/sst_forecast_data_2014_2017.csv"),
    (2017, "../data/processed/sst_forecast_data.csv"),
]:
    print(f"Baseline 2001–{train_end_year}")
    build_forecast_data(train_end_year, output_file)

Baseline 2001–2010
Saved 264 calendar months to sst_forecast_data_2011_2013.csv
Months with an SST anomaly: 260
Baseline 2001–2013
Saved 264 calendar months to sst_forecast_data_2014_2017.csv
Months with an SST anomaly: 260
Baseline 2001–2017
Saved 264 calendar months to sst_forecast_data.csv
Months with an SST anomaly: 260


## 3. Baseline models: one-month SST anomaly forecast (`forecast_monthly_sst.py`)

Reads `sst_forecast_data.csv` from Section 2.

### 3a. Persistence

In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

sst = pd.read_csv("../data/processed/sst_forecast_data.csv")
icen = pd.read_csv(ICEN_CSV).rename(
    columns={"Year": "year", "Month": "month"}
)

data = sst.merge(
    icen[["year", "month", "ENSO", "Episode"]],
    on=["year", "month"],
    how="left",
    validate="one_to_one",
)

data["date"] = pd.to_datetime(data[["year", "month"]].assign(day=1))
data = data.set_index("date").sort_index().asfreq("MS")

# Keep gaps so the forecast always uses the previous calendar month.
data["actual"] = data["sst_anom"].where(data["low_coverage"].eq(False))
data["persistence"] = data["actual"].shift(1)

# Use the target month's ICEN label to group the errors after forecasting.
data["phase"] = data["Episode"].where(
    data["Episode"].isin(["El Niño", "La Niña"])
)
data.loc[data["ENSO"].eq("Neutral"), "phase"] = "Neutral"

test = data.loc["2018-01":"2022-12"].dropna(
    subset=["actual", "persistence"]
).copy()
test["absolute_error"] = (test["actual"] - test["persistence"]).abs()

print(f"Test months: {len(test)}")
print(f"Overall persistence MAE: {test['absolute_error'].mean():.3f} °C")

summary = (
    test.groupby("phase")["absolute_error"]
    .agg(Months="size", MAE="mean")
    .reindex(["El Niño", "La Niña", "Neutral"])
)
print("\nPersistence by phase (MAE in °C)")
print(summary.round(3).to_string())

Test months: 58
Overall persistence MAE: 0.216 °C

Persistence by phase (MAE in °C)
         Months    MAE
phase                 
El Niño       4  0.494
La Niña      39  0.200
Neutral      15  0.184


### 3b. Linear regression on lagged SST and season

In [7]:
# For a forecast of March, lag 1 is February and lag 2 is January.
data["sst_lag1"] = data["persistence"]
data["sst_lag2"] = data["actual"].shift(2)

# Encode the target month so December and January are close together.
angle = 2 * np.pi * (data.index.month - 1) / 12
data["month_sin"] = np.sin(angle)
data["month_cos"] = np.cos(angle)

features = ["sst_lag1", "sst_lag2", "month_sin", "month_cos"]
required = ["actual", *features]

train = data.loc[:"2017-12"].dropna(subset=required)
test_common = data.loc["2018-01":"2022-12"].dropna(subset=required).copy()

model = LinearRegression()
model.fit(train[features], train["actual"])
test_common["regression"] = model.predict(test_common[features])

print(f"\nTraining months: {len(train)}")
print(f"Shared test months: {len(test_common)}")
for name in ["persistence", "regression"]:
    mae = (test_common["actual"] - test_common[name]).abs().mean()
    print(f"{name.title()} MAE: {mae:.3f} °C")


Training months: 190
Shared test months: 58
Persistence MAE: 0.216 °C
Regression MAE: 0.196 °C


### 3c. Regression with lagged ICEN

In [8]:
data = data.join(icen.set_index(["year", "month"])["ICEN"], on=["year", "month"])

# ICEN is a centred 3-month index. Allow an extra month for publication.
# For a March forecast, this uses the ICEN value labelled December.
data["icen_lag3"] = data["ICEN"].shift(3)

features_with_icen = [*features, "icen_lag3"]
required = ["actual", *features_with_icen]

train_apped = data.loc[:"2017-12"].dropna(subset=required)
test_apped = data.loc["2018-01":"2022-12"].dropna(subset=required).copy()

print(f"\nTest months shared by all models: {len(test_apped)}")
mae = (test_apped["actual"] - test_apped["persistence"]).abs().mean()
print(f"Persistence MAE: {mae:.3f} °C")

for label, inputs in [
    ("Regression without ICEN", features),
    ("Regression with ICEN", features_with_icen),
]:
    model = LinearRegression().fit(train_apped[inputs], train_apped["actual"])
    prediction = model.predict(test_apped[inputs])
    mae = (test_apped["actual"] - prediction).abs().mean()
    print(f"{label} MAE: {mae:.3f} °C")


Test months shared by all models: 58
Persistence MAE: 0.216 °C
Regression without ICEN MAE: 0.196 °C
Regression with ICEN MAE: 0.195 °C


### 3d. XGBoost, and errors by ENSO phase

In [9]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42,
)

xgb.fit(train_apped[features_with_icen], train_apped["actual"])
test_apped["xgboost"] = xgb.predict(test_apped[features_with_icen])

mae = (test_apped["actual"] - test_apped["xgboost"]).abs().mean()
print(f"XGBoost with ICEN MAE: {mae:.3f} °C")

test_apped["regression_with_icen"] = model.predict(
    test_apped[features_with_icen]
)

predictions = test_apped[
    ["persistence", "regression_with_icen", "xgboost"]
].rename(columns={"regression_with_icen": "regression"})

errors = predictions.sub(test_apped["actual"], axis=0)

overall = pd.DataFrame({
    "MAE (°C)": errors.abs().mean(),
    "RMSE (°C)": errors.pow(2).mean().pow(0.5),
})

print(f"\nOverall one-month forecast errors ({len(errors)} test months)")
print(overall.round(3).to_string())

for phase in ["El Niño", "La Niña", "Neutral"]:
    phase_errors = errors.loc[test_apped["phase"].eq(phase)]

    summary = pd.DataFrame({
        "MAE (°C)": phase_errors.abs().mean(),
        "RMSE (°C)": phase_errors.pow(2).mean().pow(0.5),
    })

    print(f"\n{phase} ({len(phase_errors)} test months)")
    print(summary.round(3).to_string())

XGBoost with ICEN MAE: 0.195 °C

Overall one-month forecast errors (58 test months)
             MAE (°C)  RMSE (°C)
persistence     0.216      0.274
regression      0.195      0.254
xgboost         0.195      0.251

El Niño (4 test months)
             MAE (°C)  RMSE (°C)
persistence     0.494      0.564
regression      0.454      0.517
xgboost         0.411      0.478

La Niña (39 test months)
             MAE (°C)  RMSE (°C)
persistence     0.200      0.248
regression      0.183      0.232
xgboost         0.191      0.242

Neutral (15 test months)
             MAE (°C)  RMSE (°C)
persistence     0.184      0.213
regression      0.158      0.195
xgboost         0.146      0.175


## 4. Baseline models: remaining episode duration (`forecast_episode_duration.py`)

Compares the historical median, linear regression, ARIMA, ETS and BSTS. Uses ICEN only. Episodes follow the ±0.4 °C, 3-month rule from the setup cell.

In [10]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA

ICEN_DELAY = 2
MIN_TRAIN_EPISODES = 3
MIN_ICEN_MONTHS = 24
ARIMA_ORDER = (1, 0, 1)
MAX_ARIMA_STEPS = 36

STUDY_START = pd.Period("2001-01", freq="M")
STUDY_END = pd.Period("2022-12", freq="M")

FEATURES = ["ICEN at confirmation"]
TARGET = "Actual months left"
DURATION_METHODS = ["Median", "Regression", "ARIMA", "ETS", "BSTS"]

# ---- ETS and BSTS duration forecasts (added) -------------------------------------
# Both forecast the monthly ICEN path from the history available on the issue date.
# Neither pulls back to normal on its own (BSTS's level is a random walk), so instead of one
# average path they simulate N_PATHS possible futures. The episode is predicted to end at the
# median month in which the simulated ICEN first leaves the episode range (below ENSO_WARM for
# El Niño, above ENSO_COLD for La Niña). If the median path never leaves within the window,
# the forecast is capped at the window end and still scored.
import warnings
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

N_PATHS = 1000
BSTS_DRAWS, BSTS_BURN = 3000, 1000


def in_phase(values, phase):
    return values >= ENSO_WARM if phase == "El Niño" else values <= ENSO_COLD


def months_left_from_paths(paths, phase, forecast_month):
    """paths: DataFrame indexed by future months, one column per simulated path."""
    outside = ~in_phase(paths.to_numpy(), phase)
    first_exit = np.where(outside.any(axis=0), outside.argmax(axis=0), np.inf)
    median_exit = np.quantile(first_exit, 0.5, method="lower")
    if np.isinf(median_exit):
        # The median path never leaves the episode range: the model says the episode lasts at
        # least to the end of the window, so the forecast is capped there and still scored.
        return max(0, paths.index[-1].ordinal - forecast_month.ordinal), "Scored (capped at window end)"
    last_episode_month = paths.index[int(median_exit)] - 1
    return max(0, last_episode_month.ordinal - forecast_month.ordinal), "Scored"


def ets_paths(history, steps, seed):
    """ETS with additive errors; no trend, additive or damped trend chosen by AIC; simulated paths."""
    fits = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for trend, damped in [(None, False), ("add", False), ("add", True)]:
            try:
                fits.append(ETSModel(history, error="add", trend=trend, damped_trend=damped).fit(disp=False))
            except (ValueError, np.linalg.LinAlgError):
                continue
        best = min(fits, key=lambda f: f.aic)
        sims = best.simulate(nsimulations=steps, repetitions=N_PATHS, anchor="end",
                              rng=np.random.default_rng(seed))
    future = pd.period_range(history.index[-1] + 1, periods=steps, freq="M")
    return pd.DataFrame(np.asarray(sims), index=future)


def bsts_paths(history, steps, seed, draws=BSTS_DRAWS, burn=BSTS_BURN):
    """Local-level BSTS fitted by Gibbs sampling (bsts-style default priors); posterior predictive paths.

    y_t = mu_t + e_t, e_t ~ N(0, s_e^2);  mu_t = mu_{t-1} + u_t, u_t ~ N(0, s_u^2)
    """
    rng = np.random.default_rng(seed)
    y = history.to_numpy(float); n = len(y); sdy = y.std(ddof=1)
    a0, se_b, su_b = 0.005, 0.005 * sdy ** 2, 0.005 * (0.01 * sdy) ** 2
    se2, su2 = sdy ** 2 / 4, (0.1 * sdy) ** 2
    kept = draws - burn
    keep_every = max(1, kept // N_PATHS)
    paths = []
    m, P = np.empty(n), np.empty(n)
    for it in range(draws):
        # Forward filter
        m_prev, P_prev = y[0], sdy ** 2
        for t in range(n):
            Pp = P_prev + (su2 if t else 0.0)
            K = Pp / (Pp + se2)
            m_prev = m_prev + K * (y[t] - m_prev)
            P_prev = (1 - K) * Pp
            m[t], P[t] = m_prev, P_prev
        # Backward sample the level
        z = rng.standard_normal(n)
        mu = np.empty(n)
        mu[-1] = m[-1] + np.sqrt(P[-1]) * z[-1]
        for t in range(n - 2, -1, -1):
            J = P[t] / (P[t] + su2)
            mu[t] = m[t] + J * (mu[t + 1] - m[t]) + np.sqrt((1 - J) * P[t]) * z[t]
        resid = y - mu
        se2 = 1 / rng.gamma(a0 + n / 2, 1 / (se_b + resid @ resid / 2))
        d = np.diff(mu)
        su2 = min(1 / rng.gamma(a0 + (n - 1) / 2, 1 / (su_b + d @ d / 2)), sdy ** 2)
        if it >= burn and (it - burn) % keep_every == 0:
            level = mu[-1] + np.cumsum(rng.normal(0, np.sqrt(su2), steps))
            paths.append(level + rng.normal(0, np.sqrt(se2), steps))
    future = pd.period_range(history.index[-1] + 1, periods=steps, freq="M")
    return pd.DataFrame(np.column_stack(paths), index=future)


def load_icen():
    icen = pd.read_csv(ICEN_CSV)
    dates = pd.to_datetime(
        icen[["Year", "Month"]]
        .rename(columns={"Year": "year", "Month": "month"})
        .assign(day=1)
    )
    icen["month"] = dates.dt.to_period("M")
    return icen.sort_values("month")


def build_episode_table(icen):
    episodes = []

    for episode_id, event in icen.dropna(subset=["Episode_ID"]).groupby("Episode_ID"):
        start = event["month"].iloc[0]
        end = event["month"].iloc[-1]

        if start < STUDY_START or end > STUDY_END:
            continue

        # The third qualifying ICEN value confirms the episode.
        confirmation = event.iloc[2]
        forecast_month = confirmation["month"] + ICEN_DELAY

        episodes.append({
            "Episode_ID": int(episode_id),
            "Phase": confirmation["Episode"],
            "Start": start,
            "End": end,
            "Forecast month": forecast_month,
            "End known month": end + 1 + ICEN_DELAY,
            "ICEN at confirmation": float(confirmation["ICEN"]),
            TARGET: max(0, end.ordinal - forecast_month.ordinal),
        })

    return pd.DataFrame(episodes).sort_values("Forecast month").reset_index(drop=True)


def make_forecasts(results, icen):
    series = icen.set_index("month")["ICEN"]
    calendar = pd.period_range(series.index.min(), series.index.max(), freq="M")
    series = series.reindex(calendar)

    results["Past episodes used"] = 0
    results["Median months left"] = float("nan")
    results["Regression months left"] = float("nan")
    results["ARIMA months left"] = float("nan")
    results["ARIMA status"] = "Insufficient ICEN history"
    for method in ["ETS", "BSTS"]:
        results[f"{method} months left"] = float("nan")
        results[f"{method} status"] = "Insufficient ICEN history"

    for i, episode in results.iterrows():
        forecast_month = episode["Forecast month"]
        phase = episode["Phase"]

        # The benchmarks learn from same-phase episodes with known end dates.
        past = results.loc[
            results["Phase"].eq(phase)
            & results["End known month"].le(forecast_month)
        ]
        results.loc[i, "Past episodes used"] = len(past)

        if not past.empty:
            results.loc[i, "Median months left"] = past[TARGET].median()

        if len(past) >= MIN_TRAIN_EPISODES:
            model = LinearRegression().fit(past[FEATURES], past[TARGET])
            prediction = model.predict(results.loc[[i], FEATURES])[0]
            results.loc[i, "Regression months left"] = max(0.0, prediction)

        # At the issue date, only ICEN through the confirmation month is available.
        last_available_month = forecast_month - ICEN_DELAY
        history = series.loc[:last_available_month]

        if history.count() < MIN_ICEN_MONTHS:
            continue

        # ETS and BSTS: median exit month over simulated ICEN paths
        for method, simulate in [("ETS", ets_paths), ("BSTS", bsts_paths)]:
            paths = simulate(history, MAX_ARIMA_STEPS, seed=int(episode["Episode_ID"]))
            left, status = months_left_from_paths(paths, phase, forecast_month)
            results.loc[i, f"{method} months left"] = left
            results.loc[i, f"{method} status"] = status

        fitted = ARIMA(history, order=ARIMA_ORDER).fit()
        if not fitted.mle_retvals.get("converged", True):
            results.loc[i, "ARIMA status"] = "Fit did not converge"
            continue

        path = fitted.forecast(steps=MAX_ARIMA_STEPS)
        outside_phase = ~in_phase(path, phase)   # ±0.4 °C rule (was 0.5 / -0.7)
        exit_months = path.index[outside_phase]

        if exit_months.empty:
            results.loc[i, "ARIMA status"] = "No exit within forecast window"
            continue

        # Count only episode months after the forecast month.
        last_episode_month = exit_months[0] - 1
        results.loc[i, "ARIMA months left"] = max(
            0, last_episode_month.ordinal - forecast_month.ordinal
        )
        results.loc[i, "ARIMA status"] = "Scored"

    for method in DURATION_METHODS:
        results[f"{method} error"] = (
            results[f"{method} months left"] - results[TARGET]
        ).abs()

    return results


def main():
    icen = load_icen()
    results = make_forecasts(build_episode_table(icen), icen)

    print("\nDuration forecasts by episode")
    print(
        results.drop(columns="End known month").to_string(
            index=False, float_format="{:.2f}".format, na_rep="—"
        )
    )

    print("\nForecast status by method")
    status_cols = {m: results[f"{m} status"].value_counts() for m in ["ARIMA", "ETS", "BSTS"]}
    print(pd.DataFrame(status_cols).fillna(0).astype(int).to_string())

    # Score all methods on exactly the same episodes.
    shared = results.dropna(subset=[f"{m} error" for m in DURATION_METHODS])

    print("\nDuration MAE and RMSE in months on the same episodes")
    if shared.empty:
        print("No episodes had forecasts from all methods.")
        return

    scores = {"Episodes": shared.groupby("Phase")["Episode_ID"].size()}
    for m in DURATION_METHODS:
        scores[f"{m}_MAE"] = shared.groupby("Phase")[f"{m} error"].mean()
        scores[f"{m}_RMSE"] = shared.groupby("Phase")[f"{m} error"].agg(lambda e: e.pow(2).mean() ** 0.5)
    summary = pd.DataFrame(scores)
    summary.loc["All"] = [len(shared)] + [
        shared[f"{m} error"].mean() if k == "MAE" else shared[f"{m} error"].pow(2).mean() ** 0.5
        for m in DURATION_METHODS for k in ["MAE", "RMSE"]]
    print(summary.to_string(float_format="{:.2f}".format))

In [11]:
main()

.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params



Duration forecasts by episode
 Episode_ID   Phase   Start     End Forecast month  ICEN at confirmation  Actual months left  Past episodes used  Median months left  Regression months left  ARIMA months left                   ARIMA status  ETS months left                    ETS status  BSTS months left                   BSTS status  Median error  Regression error  ARIMA error  ETS error  BSTS error
          1 La Niña 2001-06 2002-01        2001-10                 -0.88                   3                   0                   —                       —                  —      Insufficient ICEN history                —     Insufficient ICEN history                 —     Insufficient ICEN history             —                 —            —          —           —
          2 El Niño 2002-03 2002-05        2002-07                  0.67                   0                   0                   —                       —                  —      Insufficient ICEN history                —     I

## 5. Peak SST over the rest of an episode: ARIMA + GAM vs. persistence and historical median (`forecast_peak_sst_arima_gam.py`)

Uses `build_episode_table`, `load_icen`, `make_forecasts` and `ICEN_DELAY` from Section 4. Reads all three `sst_forecast_data*.csv` files from Section 2.

`FEATURES` is renamed `PEAK_FEATURES` here, because Section 4's `make_forecasts` reads the global `FEATURES`. This cell redefines `main`, so re-run Section 4's code cell before calling its `main()` again.

In [12]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

FIRST_FORECAST = pd.Period("2011-01", freq="M")
MIDDLE_PERIOD_START = pd.Period("2014-01", freq="M")
LATER_PERIOD_START = pd.Period("2018-01", freq="M")

MIN_TRAIN_ROWS = 36
LINEAR_FEATURES = ["latest_sst", "previous_sst", "month_sin", "month_cos"]
PEAK_FEATURES = [*LINEAR_FEATURES, "available_icen"]


def load_sst(filename):
    data = pd.read_csv(filename)
    dates = pd.to_datetime(data[["year", "month"]].assign(day=1))
    data.index = pd.PeriodIndex(dates, freq="M")

    sst = data["sst_anom"].where(data["low_coverage"].eq(False)).sort_index()
    calendar = pd.period_range(sst.index.min(), sst.index.max(), freq="M")
    return sst.reindex(calendar)


def make_gam():
    terms = ColumnTransformer([
        ("linear", "passthrough", LINEAR_FEATURES),
        (
            "icen_curve",
            SplineTransformer(
                n_knots=3,
                degree=2,
                include_bias=False,
                extrapolation="constant",
            ),
            ["available_icen"],
        ),
    ])
    return make_pipeline(terms, LinearRegression())


def forecast_sst_path(sst, icen, issue_month, months_left):
    """Fit a separate GAM for each lead time from the same issue month."""
    data = pd.DataFrame({"actual": sst, "ICEN": icen.reindex(sst.index)})
    angle = 2 * np.pi * (data.index.month - 1) / 12
    data["month_sin"] = np.sin(angle)
    data["month_cos"] = np.cos(angle)

    predictions = []

    for lead in range(1, months_left + 1):
        target_month = issue_month + lead
        if target_month not in data.index:
            return None, "Forecast extends beyond SST calendar"

        pairs = data.assign(
            latest_sst=data["actual"].shift(lead),
            previous_sst=data["actual"].shift(lead + 1),
            available_icen=data["ICEN"].shift(lead + ICEN_DELAY),
        )

        # Train only on SST outcomes available by the forecast month.
        train = pairs.loc[:issue_month].dropna(subset=["actual", *PEAK_FEATURES])
        inputs = pairs.loc[[target_month], PEAK_FEATURES]

        if len(train) < MIN_TRAIN_ROWS:
            return None, f"Too few training months at lead {lead}"
        if inputs.isna().any().any():
            return None, "Missing SST or ICEN at forecast date"

        gam = make_gam().fit(train[PEAK_FEATURES], train["actual"])
        predictions.append(gam.predict(inputs)[0])

    return np.asarray(predictions), "Forecast issued"


def observed_extreme(sst, issue_month, end_month, extreme):
    """Calculate the observed extreme over the remaining episode months."""
    if end_month <= issue_month:
        return np.nan

    months = pd.period_range(issue_month + 1, end_month, freq="M")
    observed = sst.reindex(months)

    if observed.isna().any():
        return np.nan

    return extreme(observed.to_numpy())


def historical_median(episodes, sst, phase, issue_month, extreme):
    # Only use earlier same-phase episodes whose end was already known.
    past = episodes.loc[
        episodes["Phase"].eq(phase)
        & episodes["End known month"].le(issue_month)
        & episodes["Actual months left"].gt(0)
    ]

    extremes = [
        observed_extreme(sst, event["Forecast month"], event["End"], extreme)
        for _, event in past.iterrows()
    ]
    usable = pd.Series(extremes, dtype=float).dropna()
    return usable.median(), len(usable)


def print_scores(results, forecast_columns, title):
    common = results.dropna(
        subset=["Actual extreme", *forecast_columns.values()]
    )

    print(f"\n{title} (°C)")
    if common.empty:
        print("No episodes had a usable observed extreme and all forecasts.")
        return

    scores = []

    for phase, cases in common.groupby("Phase"):
        for model, column in forecast_columns.items():
            errors = cases[column] - cases["Actual extreme"]
            scores.append({
                "Phase": phase,
                "Model": model,
                "Episodes": len(cases),
                "MAE": errors.abs().mean(),
                "RMSE": np.sqrt(errors.pow(2).mean()),
            })

    print(pd.DataFrame(scores).to_string(
        index=False, float_format="{:.3f}".format
    ))


def main():
    icen = load_icen()
    episodes = make_forecasts(build_episode_table(icen), icen)
    icen_series = icen.set_index("month")["ICEN"]

    sst_2011_2013 = load_sst("../data/processed/sst_forecast_data_2011_2013.csv")
    sst_2014_2017 = load_sst("../data/processed/sst_forecast_data_2014_2017.csv")
    sst_2018_2022 = load_sst("../data/processed/sst_forecast_data.csv")

    rows = []

    for _, episode in episodes.iterrows():
        issue = episode["Forecast month"]
        if issue < FIRST_FORECAST:
            continue

        # Choose an SST reference fitted before the forecast period began.
        if issue < MIDDLE_PERIOD_START:
            sst = sst_2011_2013
        elif issue < LATER_PERIOD_START:
            sst = sst_2014_2017
        else:
            sst = sst_2018_2022

        phase = episode["Phase"]
        extreme = np.max if phase == "El Niño" else np.min
        predicted_left = episode["ARIMA months left"]

        median, history_count = historical_median(
            episodes, sst, phase, issue, extreme
        )

        row = {
            "Episode_ID": episode["Episode_ID"],
            "Phase": phase,
            "Forecast month": issue,
            "Actual months left": int(episode["Actual months left"]),
            "ARIMA months left": predicted_left,
            "Actual extreme": np.nan,
            "GAM extreme": np.nan,
            "Persistence extreme": sst.get(issue, np.nan),
            "Historical median extreme": median,
            "Historical episodes used": history_count,
            "Status": "No ARIMA duration forecast",
        }

        if pd.notna(predicted_left):
            if predicted_left == 0:
                row["Status"] = "Predicted episode already over"
            else:
                path, status = forecast_sst_path(
                    sst, icen_series, issue, int(predicted_left)
                )
                row["Status"] = status
                if path is not None:
                    row["GAM extreme"] = extreme(path)

        # Use the actual end date only to calculate the observed outcome.
        row["Actual extreme"] = observed_extreme(
            sst, issue, episode["End"], extreme
        )
        rows.append(row)

    results = pd.DataFrame(rows)
    if results.empty:
        print("No episodes in the SST forecast period.")
        return

    print("\nRemaining SST extremes by episode (°C)")
    print(results.to_string(
        index=False, float_format="{:.3f}".format, na_rep="—"
    ))

    continued = results["Actual months left"].gt(0)
    gam_issued = results["GAM extreme"].notna()
    predicted_zero = results["ARIMA months left"].eq(0)

    print("\nForecast coverage")
    print(f"Episodes that actually continued: {continued.sum()}")
    print(
        "Predicted zero months left but continued: "
        f"{(predicted_zero & continued).sum()}"
    )
    print(
        "Any missing GAM forecast despite continuation: "
        f"{(~gam_issued & continued).sum()}"
    )
    print(
        "GAM forecast issued but no months actually remained: "
        f"{(gam_issued & ~continued).sum()}"
    )
    print(
        "Continuing episodes with incomplete observed SST: "
        f"{(continued & results['Actual extreme'].isna()).sum()}"
    )

    original_models = {
        "ARIMA + GAM": "GAM extreme",
        "Persistence": "Persistence extreme",
    }
    print_scores(results, original_models, "ARIMA + GAM vs. persistence")

    print_scores(
        results,
        {**original_models, "Historical median": "Historical median extreme"},
        "Three-way comparison on shared episodes",
    )

In [13]:
main()

.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params



Remaining SST extremes by episode (°C)
 Episode_ID   Phase Forecast month  Actual months left  ARIMA months left  Actual extreme  GAM extreme  Persistence extreme  Historical median extreme  Historical episodes used          Status
         13 La Niña        2012-01                   0              5.000               —        0.122                0.309                     -0.480                         4 Forecast issued
         14 El Niño        2012-07                   0              3.000               —        0.487                0.629                      0.112                         2 Forecast issued
         15 La Niña        2013-03                   8              7.000          -0.364       -0.206               -0.192                     -0.480                         4 Forecast issued
         16 El Niño        2014-09                   2              3.000           0.503        0.180                0.051                      0.109                         2 Forecast is